In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
pip install transformers datasets torchaudio

Note: you may need to restart the kernel to use updated packages.


In [3]:
from transformers import pipeline

asr = pipeline("automatic-speech-recognition", model="openai/whisper-small")

result = asr("amharic_audio.wav")
print(result["text"])

'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/openai/whisper-small/resolve/main/config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

texts, labels = zip(*data)

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

model = LogisticRegression()
model.fit(X, labels)

In [ ]:
text = asr("amharic_audio.wav")["text"]

X_test = vectorizer.transform([text])
intent = model.predict(X_test)

print(intent)

In [ ]:
pip install sounddevice

In [ ]:
import sounddevice as sd
import queue

q = queue.Queue()

def callback(indata, frames, time, status):
    q.put(indata.copy())

stream = sd.InputStream(callback=callback)
stream.start()

In [ ]:
buffer = []

while True:
    chunk = q.get()
    buffer.append(chunk)

    if len(buffer) > 20:  # ~1 second of audio
        audio_data = combine(buffer)

        text = asr(audio_data)["text"]
        intent = model.predict(vectorizer.transform([text]))

        print("TEXT:", text)
        print("INTENT:", intent)

        buffer = []